# Parquet Preparation

In [1]:
from pathlib import Path

import pandas as pd
import requests


# Minimum working request
url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": 51.0447,
    "longitude": -114.0719,
    "start_date": "2025-01-01",
    "end_date": "2025-01-02",
    "hourly": "temperature_2m,precipitation",
    "models": "era5",
    "timezone": "GMT",
    "timeformat": "unixtime",
}

response = requests.get(
    url,
    params=params,
    timeout=60,
)
response.raise_for_status()

df = pd.DataFrame(
    response.json()["hourly"]
)

df["time"] = pd.to_datetime(
    df["time"],
    unit="s",
    utc=True,
)

df.head()


,time,temperature_2m,precipitation
0,2025-01-01 00:00:00+00:00,-7.7,0.1
1,2025-01-01 01:00:00+00:00,-8.3,0.1
2,2025-01-01 02:00:00+00:00,-8.9,0.1
3,2025-01-01 03:00:00+00:00,-9.1,0.2
4,2025-01-01 04:00:00+00:00,-9.4,0.2


In [2]:
# Parquet stores column types instead of reducing them to text.
sample_path = Path(
    "sample_weather.parquet"
)

df.to_parquet(
    sample_path,
    index=False,
)

sample = pd.read_parquet(
    sample_path
)

print(df.dtypes)
print()
print(sample.dtypes)
print()
print(
    "Equal:",
    sample.equals(df),
)


time              datetime64[s, UTC]
temperature_2m               float64
precipitation                float64
dtype: object

time              datetime64[ms, UTC]
temperature_2m                float64
precipitation                 float64
dtype: object

Equal: False


In [3]:
import pyarrow.parquet as pq


parquet_file = pq.ParquetFile(
    sample_path
)

print(parquet_file.schema)
print()
print(
    "Rows:",
    parquet_file.metadata.num_rows,
)
print(
    "Row groups:",
    parquet_file.metadata.num_row_groups,
)

pd.read_parquet(
    sample_path,
    columns=[
        "time",
        "temperature_2m",
    ],
).head()


required group field_id=-1 schema {
  optional int64 field_id=-1 time (Timestamp(isAdjustedToUTC=true, timeUnit=milliseconds, is_from_converted_type=false, force_set_converted_type=false));
  optional double field_id=-1 temperature_2m;
  optional double field_id=-1 precipitation;
}


Rows: 48
Row groups: 1


,time,temperature_2m
0,2025-01-01 00:00:00+00:00,-7.7
1,2025-01-01 01:00:00+00:00,-8.3
2,2025-01-01 02:00:00+00:00,-8.9
3,2025-01-01 03:00:00+00:00,-9.1
4,2025-01-01 04:00:00+00:00,-9.4


In [4]:
sample_path.unlink()

In [5]:
import time


# Configuration
URL = "https://archive-api.open-meteo.com/v1/archive"

LATITUDE = 51.0447
LONGITUDE = -114.0719
MODEL = "era5"
LOCAL_TIMEZONE = "America/Edmonton"

HOURLY_VARIABLES = [
    "temperature_2m",
    "relative_humidity_2m",
    "dew_point_2m",
    "precipitation",
    "rain",
    "snowfall",
    "weather_code",
    "cloud_cover",
    "pressure_msl",
    "surface_pressure",
    "wind_speed_10m",
    "wind_direction_10m",
    "shortwave_radiation",
    "vapour_pressure_deficit",
    "soil_temperature_0_to_7cm",
]

START_LOCAL = pd.Timestamp(
    "2000-01-01 00:00",
    tz=LOCAL_TIMEZONE,
)

END_LOCAL = pd.Timestamp(
    "2026-10-01 00:00",
    tz=LOCAL_TIMEZONE,
)

REQUEST_DELAY_SECONDS = 4

DATA_PATH = Path(
    "data"
) / "weather.parquet"


In [6]:
# Reused from the CSV update.
def create_year_batches(
    start_local,
    end_local,
):
    batches = []
    batch_start = start_local

    while batch_start < end_local:
        next_year = pd.Timestamp(
            year=batch_start.year + 1,
            month=1,
            day=1,
            tz=batch_start.tz,
        )

        batch_end = min(
            next_year,
            end_local,
        )

        batches.append(
            (batch_start, batch_end)
        )

        batch_start = batch_end

    return batches


In [7]:
def fetch_weather_batch(
    start_local,
    end_local,
):
    start_utc = start_local.tz_convert(
        "UTC"
    )
    end_utc = end_local.tz_convert(
        "UTC"
    )

    # API accepts dates, so request the UTC envelope then trim.
    params = {
        "latitude": LATITUDE,
        "longitude": LONGITUDE,
        "start_date": (
            start_utc.date().isoformat()
        ),
        "end_date": (
            end_utc
            - pd.Timedelta(hours=1)
        ).date().isoformat(),
        "hourly": ",".join(
            HOURLY_VARIABLES
        ),
        "models": MODEL,
        "timezone": "GMT",
        "timeformat": "unixtime",
    }

    response = requests.get(
        URL,
        params=params,
        timeout=60,
    )
    response.raise_for_status()

    batch_df = pd.DataFrame(
        response.json()["hourly"]
    )

    batch_df["time"] = (
        pd.to_datetime(
            batch_df["time"],
            unit="s",
            utc=True,
        )
    )

    return (
        batch_df.loc[
            batch_df["time"].ge(
                start_utc
            )
            & batch_df["time"].lt(
                end_utc
            ),
            ["time", *HOURLY_VARIABLES],
        ]
        .reset_index(drop=True)
    )


In [8]:
def fetch_weather(
    start_local,
    end_local,
):
    batches = create_year_batches(
        start_local,
        end_local,
    )

    frames = []

    for i, (
        batch_start,
        batch_end,
    ) in enumerate(
        batches,
        start=1,
    ):
        print(
            f"Request {i}/{len(batches)}: "
            f"{batch_start} -> {batch_end}"
        )

        frames.append(
            fetch_weather_batch(
                batch_start,
                batch_end,
            )
        )

        if i < len(batches):
            time.sleep(
                REQUEST_DELAY_SECONDS
            )

    df = pd.concat(
        frames,
        ignore_index=True,
    )

    expected_time = pd.date_range(
        start=start_local.tz_convert(
            "UTC"
        ),
        end=end_local.tz_convert(
            "UTC"
        ),
        freq="h",
        inclusive="left",
    )

    if not pd.DatetimeIndex(
        df["time"]
    ).equals(expected_time):
        raise ValueError(
            "Unexpected hourly coverage."
        )

    return df


In [9]:
# Full preparation
df = fetch_weather(
    START_LOCAL,
    END_LOCAL,
)

DATA_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

df.to_parquet(
    DATA_PATH,
    index=False,
)

print(f"Rows: {len(df):,}")
print(
    f"Range: {df['time'].min()} "
    f"to {df['time'].max()}"
)
print(f"Saved: {DATA_PATH}")


Request 1/27: 2000-01-01 00:00:00-07:00 -> 2001-01-01 00:00:00-07:00
Request 2/27: 2001-01-01 00:00:00-07:00 -> 2002-01-01 00:00:00-07:00
Request 3/27: 2002-01-01 00:00:00-07:00 -> 2003-01-01 00:00:00-07:00
Request 4/27: 2003-01-01 00:00:00-07:00 -> 2004-01-01 00:00:00-07:00
Request 5/27: 2004-01-01 00:00:00-07:00 -> 2005-01-01 00:00:00-07:00
Request 6/27: 2005-01-01 00:00:00-07:00 -> 2006-01-01 00:00:00-07:00
Request 7/27: 2006-01-01 00:00:00-07:00 -> 2007-01-01 00:00:00-07:00
Request 8/27: 2007-01-01 00:00:00-07:00 -> 2008-01-01 00:00:00-07:00
Request 9/27: 2008-01-01 00:00:00-07:00 -> 2009-01-01 00:00:00-07:00
Request 10/27: 2009-01-01 00:00:00-07:00 -> 2010-01-01 00:00:00-07:00
Request 11/27: 2010-01-01 00:00:00-07:00 -> 2011-01-01 00:00:00-07:00
Request 12/27: 2011-01-01 00:00:00-07:00 -> 2012-01-01 00:00:00-07:00
Request 13/27: 2012-01-01 00:00:00-07:00 -> 2013-01-01 00:00:00-07:00
Request 14/27: 2013-01-01 00:00:00-07:00 -> 2014-01-01 00:00:00-07:00
Request 15/27: 2014-01-01 00:

In [10]:
def main():
    df = fetch_weather(
        START_LOCAL,
        END_LOCAL,
    )

    DATA_PATH.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    df.to_parquet(
        DATA_PATH,
        index=False,
    )

    print(f"Rows: {len(df):,}")
    print(
        f"Range: {df['time'].min()} "
        f"to {df['time'].max()}"
    )
    print(f"Saved: {DATA_PATH}")
